# Parsed and Annotated Data
## Completed by William Brannock (svv8fs)

Here we create the LIB, CORPUS, and VOCAB tables for the Parsed and Annotated Data section of the Final Project.

The class notebook here was what I used as the major guide for this part: https://www.kaggle.com/code/ontoligent/uva-ds-5001-m04-01-pipeline

# Imports

In [1]:
from glob import glob
from pathlib import Path
import re

import nltk
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import RegexpTokenizer

In [2]:
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("stopwords", quiet=True)

True

## Config

In [3]:
data_home = "."
output_dir = "output"
source_file_dir = f"{data_home}/data"
data_prefix = "constitutions"
CSV_DELIM = "|"

In [4]:
# OHCO setup
OHCO = [
    "constitution_id",
    "provision_num",
    "para_num",
    "sent_num",
    "token_num",
]

clip_pats = [
    r"^Share$"
]

# Go through source .txt files




In [5]:
source_file_list = sorted(glob(f"{source_file_dir}/*.txt"))
source_file_list[:10], len(source_file_list)

(['./data/Afghanistan_2004.txt',
  './data/Albania_2008.txt',
  './data/Algeria_2008.txt',
  './data/Andorra_1993.txt',
  './data/Angola_2010.txt',
  './data/Antigua_and_Barbuda_1981.txt',
  './data/Argentina_1994.txt',
  './data/Armenia_2005.txt',
  './data/Australia_1985.txt',
  './data/Austria_2009.txt'],
 192)

In [ ]:
# Parsing helper functions
def parse_filename(source_file_path):
    stem = Path(source_file_path).stem
    country, file_year = stem.rsplit("_", 1)
    return stem, country.replace("_", " "), int(file_year)


# Get out title, original year, and revision year from the first line of the file
def parse_title_line(line):
    title_pat = re.compile(r"^(?P<title>.*?)(?:\s+(?P<original_year>\d{4}))?(?:\s+\(rev\.\s*(?P<revision_year>\d{4})\))?\s*$")
    match = title_pat.match(line.strip())
    if not match:
        return line.strip(), pd.NA, pd.NA
    original_year = match.group("original_year")
    revision_year = match.group("revision_year")
    return (
        match.group("title").strip(),
        int(original_year) if original_year else pd.NA,
        int(revision_year) if revision_year else pd.NA,
    )


# Parse book metadata
book_data = []
for source_file_path in source_file_list:
    constitution_id, country, file_year = parse_filename(source_file_path)
    raw_title = Path(source_file_path).read_text(encoding="utf-8", errors="ignore").splitlines()[0]
    book_data.append((constitution_id, source_file_path, raw_title, country, file_year))

book_data[:5]

[('Afghanistan_2004',
  './data/Afghanistan_2004.txt',
  'Afghanistan 2004',
  'Afghanistan',
  2004),
 ('Albania_2008',
  './data/Albania_2008.txt',
  'Albania 1998 (rev. 2008)',
  'Albania',
  2008),
 ('Algeria_2008',
  './data/Algeria_2008.txt',
  'Algeria 1963 (rev. 2008)',
  'Algeria',
  2008),
 ('Andorra_1993', './data/Andorra_1993.txt', 'Andorra 1993', 'Andorra', 1993),
 ('Angola_2010', './data/Angola_2010.txt', 'Angola 2010', 'Angola', 2010)]

In [7]:
LIB = pd.DataFrame(
    book_data,
    columns=["constitution_id", "source_file_path", "raw_title", "country", "file_year"]
).set_index("constitution_id").sort_index()
LIB.head()

,source_file_path,raw_title,country,file_year
constitution_id,,,,
Afghanistan_2004,./data/Afghanistan_2004.txt,Afghanistan 2004,Afghanistan,2004
Albania_2008,./data/Albania_2008.txt,Albania 1998 (rev. 2008),Albania,2008
Algeria_2008,./data/Algeria_2008.txt,Algeria 1963 (rev. 2008),Algeria,2008
Andorra_1993,./data/Andorra_1993.txt,Andorra 1993,Andorra,1993
Angola_2010,./data/Angola_2010.txt,Angola 2010,Angola,2010


In [8]:
title_data = LIB.raw_title.apply(parse_title_line).apply(pd.Series)
title_data.columns = ["title", "original_year", "revision_year"]
LIB = LIB.join(title_data).drop("raw_title", axis=1)
LIB.head()

,source_file_path,country,file_year,title,original_year,revision_year
constitution_id,,,,,,
Afghanistan_2004,./data/Afghanistan_2004.txt,Afghanistan,2004,Afghanistan,2004.0,<NA>
Albania_2008,./data/Albania_2008.txt,Albania,2008,Albania,1998.0,2008
Algeria_2008,./data/Algeria_2008.txt,Algeria,2008,Algeria,1963.0,2008
Andorra_1993,./data/Andorra_1993.txt,Andorra,1993,Andorra,1993.0,<NA>
Angola_2010,./data/Angola_2010.txt,Angola,2010,Angola,2010.0,<NA>


# Parse Provisions

A provision here is any native structure such as parts, chapters, titles, articles, sections, explanations, clausees, etc.

In [9]:
roman = "[IVXLCDM]+"

provision_pat_list = [
    r"^Preamble$",
    r"^(?:PART|Part)\b",
    r"^(?:TITLE|Title)\b",
    r"^(?:CHAPTER|Chapter)\b",
    r"^(?:SCHEDULE|Schedule)\b",
    rf"^(?:ARTICLE|Article)\s+(?:\d+[A-Z]?|{roman})\b",
    r"^(?:SECTION|Section)\s+[A-Za-z0-9]+\b",
    r"^\d+[A-Z]?\.\s+\S+",
    r"^(?:Explanation|Transitional Provision|Final Provision|Additional Provision)\b",
]
provision_regex = "(?:" + ")|(?:".join(provision_pat_list) + ")"

# save the provision regex in the LIB 
LIB["provision_regex"] = provision_regex

# sentence splitting and tokenization setup
SENT_SPLIT_RE = re.compile(r"(?<=[.!?;:])\s+(?=[A-Z0-9\"'])")
TOKENIZER = RegexpTokenizer(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+(?:\.[0-9]+)?")


def is_blank_or_clipped(line):
    stripped = line.strip()
    return stripped == "" or any(re.search(pat, stripped) for pat in clip_pats)


def is_provision_boundary(line, provision_regex):
    return re.search(provision_regex, line.strip(), flags=re.IGNORECASE) is not None


def split_sentences(para_str):
    sentences = [s.strip() for s in SENT_SPLIT_RE.split(para_str.strip())]
    return [s for s in sentences if s]


def clean_term(token):
    term = re.sub(r"[\W_]+", "", token).lower()
    return term or pd.NA

In [10]:
def parse_provisions(source_file_path, provision_regex):
    raw_lines = Path(source_file_path).read_text(encoding="utf-8", errors="ignore").splitlines()
    constitution_id, _country, _file_year = parse_filename(source_file_path)

    provisions = []
    current = None
    provision_num = 0

    def close_current():
        nonlocal current
        if current is not None:
            current["provision_str"] = "\n".join(current.pop("lines")).strip()
            if current["provision_str"]:
                provisions.append(current)
            current = None

    def open_provision():
        nonlocal current, provision_num
        close_current()
        provision_num += 1
        current = {
            "constitution_id": constitution_id,
            "provision_num": provision_num,
            "lines": [],
        }

    for raw_line in raw_lines[1:]:
        line = raw_line.strip()

        if is_blank_or_clipped(line):
            if current is not None:
                current["lines"].append("")
            continue

        if is_provision_boundary(line, provision_regex):
            open_provision()

        elif current is None:
            open_provision()

        current["lines"].append(raw_line.rstrip())

    close_current()
    return pd.DataFrame(provisions)

In [11]:
parse_provisions(
    LIB.loc["United_States_of_America_1992"].source_file_path,
    LIB.loc["United_States_of_America_1992"].provision_regex,
).head(20)

,constitution_id,provision_num,provision_str
0,United_States_of_America_1992,1,Preamble\n\n\nWe the People of the United Stat...
1,United_States_of_America_1992,2,Article I
2,United_States_of_America_1992,3,Section 1\n\n\nAll legislative Powers herein g...
3,United_States_of_America_1992,4,Section 2\n\n\nThe House of Representatives sh...
4,United_States_of_America_1992,5,Section 3\n\n\nThe Senate of the United States...
5,United_States_of_America_1992,6,"Section 4\n\n\nThe Times, Places and Manner of..."
6,United_States_of_America_1992,7,Section 5\n\n\nEach House shall be the Judge o...
7,United_States_of_America_1992,8,Section 6\n\n\nThe Senators and Representative...
8,United_States_of_America_1992,9,Section 7\n\n\nAll Bills for raising Revenue s...
9,United_States_of_America_1992,10,Section 8\n\n\nThe Congress shall have Power T...


# Tokenizing The Corpus

In [12]:
def tokenize_constitution(source_file_path, provision_regex):
    PROVISIONS = parse_provisions(source_file_path, provision_regex)
    tokens = []

    for provision in PROVISIONS.to_dict("records"):
        base = {
            "constitution_id": provision["constitution_id"],
            "provision_num": provision["provision_num"],
        }
        paras = [p.strip() for p in re.split(r"\n\s*\n+", provision["provision_str"]) if p.strip()]

        for para_num, para_str in enumerate(paras, start=1):
            sentence_tokens = []
            for sent_str in split_sentences(para_str):
                sent_tokens = TOKENIZER.tokenize(sent_str)
                if sent_tokens:
                    sentence_tokens.append(sent_tokens)

            if not sentence_tokens:
                continue

            tagged_sentences = nltk.pos_tag_sents(sentence_tokens, lang="eng")
            for sent_num, tagged_sentence in enumerate(tagged_sentences, start=1):
                for token_num, (token_str, pos) in enumerate(tagged_sentence, start=1):
                    tokens.append({
                        **base,
                        "para_num": para_num,
                        "sent_num": sent_num,
                        "token_num": token_num,
                        "token_str": token_str,
                        "term_str": clean_term(token_str),
                        "pos": pos,
                    })

    TOKENS = pd.DataFrame(tokens)
    TOKENS = TOKENS.dropna(subset=["term_str"])
    TOKENS = TOKENS.set_index(OHCO).sort_index()
    return TOKENS, PROVISIONS


def tokenize_collection(LIB):
    corpora = []
    provision_tables = []

    for constitution_id in LIB.index:
        print("Tokenizing", constitution_id, LIB.loc[constitution_id].title)
        source_file_path = LIB.loc[constitution_id].source_file_path
        provision_regex = LIB.loc[constitution_id].provision_regex
        TOKENS, PROVISIONS = tokenize_constitution(source_file_path, provision_regex)
        corpora.append(TOKENS)
        provision_tables.append(PROVISIONS)

    CORPUS = pd.concat(corpora).sort_index()
    PROVISIONS = pd.concat(provision_tables, ignore_index=True)
    print("Done")
    return CORPUS, PROVISIONS

In [13]:
CORPUS, PROVISIONS = tokenize_collection(LIB)
CORPUS.head()

Tokenizing Afghanistan_2004 Afghanistan
Tokenizing Albania_2008 Albania
Tokenizing Algeria_2008 Algeria
Tokenizing Andorra_1993 Andorra
Tokenizing Angola_2010 Angola
Tokenizing Antigua_and_Barbuda_1981 Antigua and Barbuda
Tokenizing Argentina_1994 Argentina 1853 (reinst. 1983, rev. 1994)
Tokenizing Armenia_2005 Armenia
Tokenizing Australia_1985 Australia
Tokenizing Austria_2009 Austria 1920 (reinst. 1945, rev. 2009)
Tokenizing Azerbaijan_2009 Azerbaijan
Tokenizing Bahamas_2002 Bahamas
Tokenizing Bahrain_2002 Bahrain
Tokenizing Bangladesh_2011 Bangladesh 1972 (reinst. 1986, rev. 2011)
Tokenizing Barbados_2007 Barbados
Tokenizing Belarus_2004 Belarus
Tokenizing Belgium_2012 Belgium
Tokenizing Belize_2001 Belize
Tokenizing Benin_1990 Benin
Tokenizing Bhutan_2008 Bhutan
Tokenizing Bolivia_2009 Bolivia
Tokenizing Bosnia_Herzegovina_2009 Bosnia-Herzegovina
Tokenizing Botswana_2002 Botswana
Tokenizing Brazil_2014 Brazil
Tokenizing Brunei_1984 Brunei
Tokenizing Bulgaria_2007 Bulgaria
Tokenizin

token_str  \
constitution_id  provision_num para_num sent_num token_num             
Afghanistan_2004 1             1        1        1          Preamble   
                               2        1        1                In   
                                                 2               the   
                                                 3              name   
                                                 4                of   

                                                            term_str pos  
constitution_id  provision_num para_num sent_num token_num                
Afghanistan_2004 1             1        1        1          preamble  JJ  
                               2        1        1                in  IN  
                                                 2               the  DT  
                                                 3              name  NN  
                                                 4                of  IN

In [14]:
CORPUS.query('constitution_id == "United_States_of_America_1992"').tail(20)

token_str  \
constitution_id               provision_num para_num sent_num token_num                
United_States_of_America_1992 61            2        1        33                  on   
                                                              34             account   
                                                              35                  of   
                                                              36                 age   
                              62            1        1        1              Section   
                                                              2                    2   
                                            2        1        1                  The   
                                                              2             Congress   
                                                              3                shall   
                                                              4                 have   
                                                              5                power   
                                                              6                   to   
                                                              7              enforce   
                                                              8                 this   
                                                              9              article   
                                                              10                  by   
                                                              11         appropriate   
                                                              12         legislation   
                                            3        1        1            Amendment   
                                                              2                XXVII   

                                                                            term_str  \
constitution_id               provision_num para_num sent_num token_num                
United_States_of_America_1992 61            2        1        33                  on   
                                                              34             account   
                                                              35                  of   
                                                              36                 age   
                              62            1        1        1              section   
                                                              2                    2   
                                            2        1        1                  the   
                                                              2             congress   
                                                              3                shall   
                                                              4                 have   
                                                              5                power   
                                                              6                   to   
                                                              7              enforce   
                                                              8                 this   
                                                              9              article   
                                                              10                  by   
                                                              11         appropriate   
                                                              12         legislation   
                                            3        1        1            amendment   
                                                              2                xxvii   

                                                                         pos  
constitution_id               provision_num para_num sent_num token_num       
United_States_of_America_1992 61           

# Extract LIB Features

In [15]:
LIB["constitution_len"] = CORPUS.groupby("constitution_id").term_str.count()
LIB["n_provisions"] = PROVISIONS.groupby("constitution_id").provision_num.count()
LIB["n_chars"] = LIB.source_file_path.apply(lambda p: len(Path(p).read_text(encoding="utf-8", errors="ignore")))
LIB.sort_values("constitution_len").head()

,source_file_path,country,file_year,title,original_year,revision_year,provision_regex,constitution_len,n_provisions,n_chars
constitution_id,,,,,,,,,,
Libya_2011,./data/Libya_2011.txt,Libya,2011,Libya,2011.0,<NA>,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,2922,6,18926
Iceland_1999,./data/Iceland_1999.txt,Iceland,1999,Iceland,1944.0,1999,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,4105,80,25334
Monaco_2002,./data/Monaco_2002.txt,Monaco,2002,Monaco,1962.0,2002,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,4425,12,28323
Latvia_2007,./data/Latvia_2007.txt,Latvia,2007,"Latvia 1922 (reinst. 1991, rev. 2007)",NaN,NaN,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,4789,125,29797
Japan_1946,./data/Japan_1946.txt,Japan,1946,Japan,1946.0,<NA>,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,4989,115,31269


# Extract VOCAB

In [16]:
empty_term_mask = CORPUS.term_str.eq("")
empty_term_mask.sum()

np.int64(0)

In [17]:
CORPUS = CORPUS.loc[~empty_term_mask].copy()

In [18]:
CORPUS["pos_group"] = CORPUS.pos.str[:2]
CORPUS.head()

token_str  \
constitution_id  provision_num para_num sent_num token_num             
Afghanistan_2004 1             1        1        1          Preamble   
                               2        1        1                In   
                                                 2               the   
                                                 3              name   
                                                 4                of   

                                                            term_str pos  \
constitution_id  provision_num para_num sent_num token_num                 
Afghanistan_2004 1             1        1        1          preamble  JJ   
                               2        1        1                in  IN   
                                                 2               the  DT   
                                                 3              name  NN   
                                                 4                of  IN   

                                                           pos_group  
constitution_id  provision_num para_num sent_num token_num            
Afghanistan_2004 1             1        1        1                JJ  
                               2        1        1                IN  
                                                 2                DT  
                                                 3                NN  
                                                 4                IN

In [19]:
VOCAB = CORPUS.term_str.value_counts().to_frame("n").sort_index()
VOCAB.index.name = "term_str"
VOCAB["n_chars"] = VOCAB.index.str.len()
VOCAB["p"] = VOCAB.n / VOCAB.n.sum()
VOCAB["i"] = -np.log2(VOCAB.p)
VOCAB.head()

,n,n_chars,p,i
term_str,,,,
0,26,1,6.051719e-06,17.334224
00,9,2,2.094826e-06,18.864738
000,210,3,4.887927e-05,14.320418
00000,2,5,4.655168e-07,21.034663
001,5,3,1.163792e-06,19.712735


# Annotate VOCAB

In [20]:
VOCAB["max_pos"] = CORPUS.value_counts(["term_str", "pos"]).unstack(fill_value=0).idxmax(1)
VOCAB["max_pos_group"] = CORPUS.value_counts(["term_str", "pos_group"]).unstack(fill_value=0).idxmax(1)
VOCAB.head()

,n,n_chars,p,i,max_pos,max_pos_group
term_str,,,,,,
0,26,1,6.051719e-06,17.334224,CD,CD
00,9,2,2.094826e-06,18.864738,CD,CD
000,210,3,4.887927e-05,14.320418,CD,CD
00000,2,5,4.655168e-07,21.034663,CD,CD
001,5,3,1.163792e-06,19.712735,CD,CD


In [21]:
VOCAB["n_pos_group"] = CORPUS.value_counts(["term_str", "pos_group"]).unstack(fill_value=0).astype("bool").sum(1)
VOCAB["cat_pos_group"] = CORPUS.value_counts(["term_str", "pos_group"]).unstack(fill_value=0).astype("bool").apply(lambda x: " ".join(x.index[x]), 1)
VOCAB["n_pos"] = CORPUS.value_counts(["term_str", "pos"]).unstack(fill_value=0).astype("bool").sum(1)
VOCAB["cat_pos"] = CORPUS.value_counts(["term_str", "pos"]).unstack(fill_value=0).astype("bool").apply(lambda x: " ".join(x.index[x]), 1)
VOCAB.head()

/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_11486/4022745382.py:1: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  VOCAB["n_pos_group"] = CORPUS.value_counts(["term_str", "pos_group"]).unstack(fill_value=0).astype("bool").sum(1)
/var/folders/2n/rqnjq39x0bl3l55y0ghnzcz00000gn/T/ipykernel_11486/4022745382.py:3: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  VOCAB["n_pos"] = CORPUS.value_counts(["term_str", "pos"]).unstack(fill_value=0).astype("bool").sum(1)


,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos
term_str,,,,,,,,,,
0,26,1,6.051719e-06,17.334224,CD,CD,1,CD,1,CD
00,9,2,2.094826e-06,18.864738,CD,CD,1,CD,1,CD
000,210,3,4.887927e-05,14.320418,CD,CD,1,CD,1,CD
00000,2,5,4.655168e-07,21.034663,CD,CD,1,CD,1,CD
001,5,3,1.163792e-06,19.712735,CD,CD,1,CD,1,CD


In [22]:
sw = pd.Series(1, index=stopwords.words("english"), name="stop")
VOCAB["stop"] = VOCAB.index.map(sw).fillna(0).astype("int")
VOCAB.head()

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop
term_str,,,,,,,,,,,
0,26,1,6.051719e-06,17.334224,CD,CD,1,CD,1,CD,0
00,9,2,2.094826e-06,18.864738,CD,CD,1,CD,1,CD,0
000,210,3,4.887927e-05,14.320418,CD,CD,1,CD,1,CD,0
00000,2,5,4.655168e-07,21.034663,CD,CD,1,CD,1,CD,0
001,5,3,1.163792e-06,19.712735,CD,CD,1,CD,1,CD,0


In [23]:
stemmer = PorterStemmer()
VOCAB["porter_stem"] = VOCAB.apply(lambda x: stemmer.stem(x.name), 1)
VOCAB.head()

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem
term_str,,,,,,,,,,,,
0,26,1,6.051719e-06,17.334224,CD,CD,1,CD,1,CD,0,0
00,9,2,2.094826e-06,18.864738,CD,CD,1,CD,1,CD,0,00
000,210,3,4.887927e-05,14.320418,CD,CD,1,CD,1,CD,0,000
00000,2,5,4.655168e-07,21.034663,CD,CD,1,CD,1,CD,0,00000
001,5,3,1.163792e-06,19.712735,CD,CD,1,CD,1,CD,0,001


# Add DFIDF


In [24]:
DOC = CORPUS.groupby("constitution_id").term_str.count().to_frame("n_tokens")
DOC["n_types"] = CORPUS.reset_index()[["constitution_id", "term_str"]].drop_duplicates().groupby("constitution_id").term_str.count()
DOC["pkr"] = DOC.n_types / DOC.n_tokens
DOC = DOC.join(LIB[["country", "file_year", "original_year", "revision_year", "title"]])
DOC.sort_values("pkr").head(20)

,n_tokens,n_types,pkr,country,file_year,original_year,revision_year,title
constitution_id,,,,,,,,
India_2012,103941,3919,0.037704,India,2012,1949.0,2012,India
St_Kitts_and_Nevis_1983,46896,2106,0.044908,St Kitts and Nevis,1983,1983.0,<NA>,St. Kitts and Nevis
Malaysia_1996,64746,3123,0.048235,Malaysia,1996,1957.0,1996,Malaysia
St_Lucia_1978,40394,1998,0.049463,St Lucia,1978,1978.0,<NA>,St. Lucia
Sweden_2012,59926,3067,0.051180,Sweden,2012,1974.0,2012,Sweden
Jamaica_1994,38422,1974,0.051377,Jamaica,1994,1962.0,1994,Jamaica
Fiji_2013,47371,2448,0.051677,Fiji,2013,2013.0,<NA>,Fiji
Singapore_2010,44342,2307,0.052027,Singapore,2010,1959.0,2010,Singapore
Lesotho_1998,42911,2267,0.052830,Lesotho,1998,1993.0,1998,Lesotho


In [25]:
N = DOC.shape[0]

DF = CORPUS.reset_index()[["constitution_id", "term_str"]].drop_duplicates().term_str.value_counts().sort_index()
IDF = np.log2(N / DF)

VOCAB["df"] = DF
VOCAB["idf"] = IDF
VOCAB["dfidf"] = VOCAB.df * VOCAB.idf
VOCAB["dp"] = VOCAB.df / N
VOCAB["di"] = np.log2(1 / VOCAB.dp)
VOCAB["dh"] = VOCAB.dp * VOCAB.di

VOCAB.sort_values(["dfidf", "n"], ascending=False).head(20)

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem,df,idf,dfidf,dp,di,dh
term_str,,,,,,,,,,,,,,,,,,
assent,623,6,0.000145,12.751575,NN,NN,3,JJ NN VB,5,JJ NN NNP VB VBD,0,assent,71,1.435215,101.900292,0.369792,1.435215,0.530731
labor,500,5,0.000116,13.068879,NN,NN,1,NN,2,NN NNP,0,labor,71,1.435215,101.900292,0.369792,1.435215,0.530731
rural,402,5,0.000094,13.383612,JJ,JJ,3,JJ NN VB,3,JJ NNP VB,0,rural,71,1.435215,101.900292,0.369792,1.435215,0.530731
hand,348,4,0.000081,13.591720,NN,NN,1,NN,2,NN NNP,0,hand,71,1.435215,101.900292,0.369792,1.435215,0.530731
servants,347,8,0.000081,13.595871,NNS,NN,2,NN VB,5,NN NNP NNS VBZ NNPS,0,servant,71,1.435215,101.900292,0.369792,1.435215,0.530731
like,303,4,0.000071,13.791489,JJ,JJ,3,JJ IN VB,3,JJ IN VB,0,like,71,1.435215,101.900292,0.369792,1.435215,0.530731
organized,299,9,0.000070,13.810662,VBN,VB,3,JJ NN VB,4,JJ NNP VBN VBD,0,organ,71,1.435215,101.900292,0.369792,1.435215,0.530731
class,291,5,0.000068,13.849788,NN,NN,1,NN,2,NN NNP,0,class,71,1.435215,101.900292,0.369792,1.435215,0.530731
fees,251,4,0.000058,14.063120,NNS,NN,1,NN,1,NNS,0,fee,71,1.435215,101.900292,0.369792,1.435215,0.530731


# Check Sample Constitutions

In [26]:
sample_ids = [
    "United_States_of_America_1992",
    "India_2012",
    "France_2008",
    "South_Africa_2012",
]

# Check how provisions are parsed
PROVISIONS.loc[PROVISIONS.constitution_id.isin(sample_ids)].head(40)

,constitution_id,provision_num,provision_str
21042,France_2008,1,Preamble\n\n\nThe French people solemnly procl...
21043,France_2008,2,"ARTICLE 1\n\n\nFrance shall be an indivisible,..."
21044,France_2008,3,Title I. ON SOVEREIGNTY
21045,France_2008,4,ARTICLE 2\n\n\nThe language of the Republic sh...
21046,France_2008,5,ARTICLE 3\n\n\nNational sovereignty shall vest...
21047,France_2008,6,ARTICLE 4\n\n\nPolitical parties and groups sh...
21048,France_2008,7,Title II. THE PRESIDENT OF THE REPUBLIC
21049,France_2008,8,ARTICLE 5\n\n\nThe President of the Republic s...
21050,France_2008,9,ARTICLE 6\n\n\nThe President of the Republic s...
21051,France_2008,10,ARTICLE 7\n\n\nThe President of the Republic s...


In [27]:
# Check Corpus structure
CORPUS.reset_index().query("constitution_id in @sample_ids").head(40)

,constitution_id,provision_num,para_num,sent_num,token_num,token_str,term_str,pos,pos_group
1220329,France_2008,1,1,1,1,Preamble,preamble,JJ,JJ
1220330,France_2008,1,2,1,1,The,the,DT,DT
1220331,France_2008,1,2,1,2,French,french,JJ,JJ
1220332,France_2008,1,2,1,3,people,people,NNS,NN
1220333,France_2008,1,2,1,4,solemnly,solemnly,RB,RB
1220334,France_2008,1,2,1,5,proclaim,proclaim,VBP,VB
1220335,France_2008,1,2,1,6,their,their,PRP$,PR
1220336,France_2008,1,2,1,7,attachment,attachment,NN,NN
1220337,France_2008,1,2,1,8,to,to,TO,TO
1220338,France_2008,1,2,1,9,the,the,DT,DT


In [28]:
# Check Vocab Structure
VOCAB.sort_values(["dfidf", "n"], ascending=False).head(20)

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,cat_pos_group,n_pos,cat_pos,stop,porter_stem,df,idf,dfidf,dp,di,dh
term_str,,,,,,,,,,,,,,,,,,
assent,623,6,0.000145,12.751575,NN,NN,3,JJ NN VB,5,JJ NN NNP VB VBD,0,assent,71,1.435215,101.900292,0.369792,1.435215,0.530731
labor,500,5,0.000116,13.068879,NN,NN,1,NN,2,NN NNP,0,labor,71,1.435215,101.900292,0.369792,1.435215,0.530731
rural,402,5,0.000094,13.383612,JJ,JJ,3,JJ NN VB,3,JJ NNP VB,0,rural,71,1.435215,101.900292,0.369792,1.435215,0.530731
hand,348,4,0.000081,13.591720,NN,NN,1,NN,2,NN NNP,0,hand,71,1.435215,101.900292,0.369792,1.435215,0.530731
servants,347,8,0.000081,13.595871,NNS,NN,2,NN VB,5,NN NNP NNS VBZ NNPS,0,servant,71,1.435215,101.900292,0.369792,1.435215,0.530731
like,303,4,0.000071,13.791489,JJ,JJ,3,JJ IN VB,3,JJ IN VB,0,like,71,1.435215,101.900292,0.369792,1.435215,0.530731
organized,299,9,0.000070,13.810662,VBN,VB,3,JJ NN VB,4,JJ NNP VBN VBD,0,organ,71,1.435215,101.900292,0.369792,1.435215,0.530731
class,291,5,0.000068,13.849788,NN,NN,1,NN,2,NN NNP,0,class,71,1.435215,101.900292,0.369792,1.435215,0.530731
fees,251,4,0.000058,14.063120,NNS,NN,1,NN,1,NNS,0,fee,71,1.435215,101.900292,0.369792,1.435215,0.530731


In [29]:
# Check LIB Strucutre
LIB.head(20)

,source_file_path,country,file_year,title,original_year,revision_year,provision_regex,constitution_len,n_provisions,n_chars
constitution_id,,,,,,,,,,
Afghanistan_2004,./data/Afghanistan_2004.txt,Afghanistan,2004,Afghanistan,2004.0,<NA>,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,10435,175,66806
Albania_2008,./data/Albania_2008.txt,Albania,2008,Albania,1998.0,2008,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,13896,623,86022
Algeria_2008,./data/Algeria_2008.txt,Algeria,2008,Algeria,1963.0,2008,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,10605,197,66590
Andorra_1993,./data/Andorra_1993.txt,Andorra,1993,Andorra,1993.0,<NA>,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,8722,295,55687
Angola_2010,./data/Angola_2010.txt,Angola,2010,Angola,2010.0,<NA>,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,26657,856,175148
Antigua_and_Barbuda_1981,./data/Antigua_and_Barbuda_1981.txt,Antigua and Barbuda,1981,Antigua and Barbuda,1981.0,<NA>,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,36749,605,214275
Argentina_1994,./data/Argentina_1994.txt,Argentina,1994,"Argentina 1853 (reinst. 1983, rev. 1994)",NaN,NaN,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,13093,170,82255
Armenia_2005,./data/Armenia_2005.txt,Armenia,2005,Armenia,1995.0,2005,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,13908,153,86487
Australia_1985,./data/Australia_1985.txt,Australia,1985,Australia,1901.0,1985,(?:^Preamble$)|(?:^(?:PART|Part)\b)|(?:^(?:TIT...,13491,174,81224


In [35]:
# Print average characters per document
print(LIB['n_chars'].mean())

138099.45833333334


# Get rows and columns for each table

In [32]:
comparison = pd.DataFrame({
    "table": ["LIB", "CORPUS", "VOCAB"],
    "n_rows": [LIB.shape[0], CORPUS.shape[0], VOCAB.shape[0]],
    "n_cols": [LIB.shape[1], CORPUS.shape[1], VOCAB.shape[1]],
})
comparison

,table,n_rows,n_cols
0,LIB,192,10
1,CORPUS,4296300,4
2,VOCAB,24868,18


# Save csvs

In [31]:
save_path = f"{output_dir}/{data_prefix}"

Path(output_dir).mkdir(exist_ok=True)
LIB.to_csv(f"{save_path}-LIB.csv", sep=CSV_DELIM)
#PROVISIONS.to_csv(f"{save_path}-PROVISIONS.csv", sep=CSV_DELIM, index=False)
CORPUS.to_csv(f"{save_path}-CORPUS.csv", sep=CSV_DELIM)
VOCAB.to_csv(f"{save_path}-VOCAB.csv", sep=CSV_DELIM)
DOC.to_csv(f"{save_path}-DOC.csv", sep=CSV_DELIM)